<a href="https://colab.research.google.com/github/devcodes28/AI-ML-Lab-24CSP503/blob/main/A_Star_and_Greedy_First_Fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq

Graph Class

In [2]:
class Graph:
    def __init__(self):
        self.adj_list = {}
        self.heuristics = {}

    def add_edge(self, from_node, to_node, cost):
        self.adj_list.setdefault(from_node, []).append((to_node, cost))
        self.adj_list.setdefault(to_node, []).append((from_node, cost))

    def set_heuristic(self, node, value):
        self.heuristics[node] = value

    def get_neighbors(self, node):
        return self.adj_list.get(node, [])

    def get_heuristic(self, node):
        return self.heuristics.get(node, float('inf'))

Create Graph and Add Edges

In [3]:
graph = Graph()

# ---- Static graph data: edges (node1, node2, cost) ----
graph.add_edge("A", "B", 1)
graph.add_edge("A", "C", 4)
graph.add_edge("B", "D", 5)
graph.add_edge("B", "E", 3)
graph.add_edge("C", "F", 2)
graph.add_edge("E", "F", 1)
graph.add_edge("D", "G", 2)
graph.add_edge("F", "G", 3)

Heuristic Values

In [4]:
# ---- Static heuristic values (estimated distance to goal G) ----
graph.set_heuristic("A", 6)
graph.set_heuristic("B", 5)
graph.set_heuristic("C", 4)
graph.set_heuristic("D", 2)
graph.set_heuristic("E", 3)
graph.set_heuristic("F", 1)
graph.set_heuristic("G", 0)

start = "A"
goal = "G"

Greedy Best First Search

In [5]:
class GreedyBestFirstSearch:

    def search(self, graph, start, goal):
        pq = []

        # Priority is based only on heuristic h(n)
        heapq.heappush(pq, (graph.get_heuristic(start), start))

        parent = {}
        visited = set()

        visited.add(start)

        print("Greedy Best First Search:")

        while pq:
            h, current = heapq.heappop(pq)

            print(f"Visiting: {current} (h={h})")

            if current == goal:
                self.print_path(parent, start, goal)
                return

            for neighbor, cost in graph.get_neighbors(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    parent[neighbor] = current

                    heapq.heappush(
                        pq,
                        (graph.get_heuristic(neighbor), neighbor)
                    )

        print("Goal not reachable.")

    def print_path(self, parent, start, goal):
        path = []
        current = goal

        while current is not None:
            path.append(current)
            current = parent.get(current)

        path.reverse()

        print("Path: " + " -> ".join(path))

In [6]:
gbfs = GreedyBestFirstSearch()
gbfs.search(graph, start, goal)

Greedy Best First Search:
Visiting: A (h=6)
Visiting: C (h=4)
Visiting: F (h=1)
Visiting: G (h=0)
Path: A -> C -> F -> G


A* Search

In [7]:
class AStarSearch:

    def search(self, graph, start, goal):
        pq = []

        # (f, node, g)
        # f(n) = g(n) + h(n)
        heapq.heappush(
            pq,
            (graph.get_heuristic(start), start, 0)
        )

        g_values = {start: 0}
        parent = {}
        visited = set()

        print("\nA* Search:")

        while pq:
            f, current, current_g = heapq.heappop(pq)

            if current in visited:
                continue

            visited.add(current)

            print(
                f"Visiting: {current} "
                f"(g={current_g}, f={f})"
            )

            if current == goal:
                self.print_path(
                    parent,
                    start,
                    goal,
                    current_g
                )
                return

            for neighbor, cost in graph.get_neighbors(current):

                new_g = current_g + cost

                if new_g < g_values.get(neighbor, float('inf')):

                    g_values[neighbor] = new_g
                    parent[neighbor] = current

                    h = graph.get_heuristic(neighbor)
                    new_f = new_g + h

                    heapq.heappush(
                        pq,
                        (new_f, neighbor, new_g)
                    )

        print("Goal not reachable.")

    def print_path(self, parent, start, goal, total_cost):
        path = []
        current = goal

        while current is not None:
            path.append(current)
            current = parent.get(current)

        path.reverse()

        print(
            "Path: "
            + " -> ".join(path)
            + f" | Total Cost: {total_cost}"
        )

In [8]:
astar = AStarSearch()
astar.search(graph, start, goal)


A* Search:
Visiting: A (g=0, f=6)
Visiting: B (g=1, f=6)
Visiting: E (g=4, f=7)
Visiting: F (g=5, f=6)
Visiting: C (g=4, f=8)
Visiting: D (g=6, f=8)
Visiting: G (g=8, f=8)
Path: A -> B -> E -> F -> G | Total Cost: 8


Cost Comparison

In [9]:
# Cost comparison

greedy_path = ["A", "C", "F", "G"]
astar_path = ["A", "B", "E", "F", "G"]


def calculate_cost(graph, path):
    total_cost = 0

    for i in range(len(path) - 1):
        current = path[i]
        next_node = path[i + 1]

        for neighbor, cost in graph.get_neighbors(current):
            if neighbor == next_node:
                total_cost += cost
                break

    return total_cost


greedy_cost = calculate_cost(graph, greedy_path)
astar_cost = calculate_cost(graph, astar_path)


print("Cost Comparison:")
print("-----------------")
print("Greedy Best First Search:")
print("Path:", " -> ".join(greedy_path))
print("Total Cost:", greedy_cost)

print("\nA* Search:")
print("Path:", " -> ".join(astar_path))
print("Total Cost:", astar_cost)

print("\nComparison:")

if greedy_cost < astar_cost:
    print("Greedy Best First Search has lower cost.")
elif astar_cost < greedy_cost:
    print("A* Search has lower cost.")
else:
    print("Both algorithms have the same cost.")

Cost Comparison:
-----------------
Greedy Best First Search:
Path: A -> C -> F -> G
Total Cost: 9

A* Search:
Path: A -> B -> E -> F -> G
Total Cost: 8

Comparison:
A* Search has lower cost.
